In [1]:
# import the necessary libraries

import ollama
import pandas as pd
import os
import time
from tqdm import tqdm

In [ ]:
# Path of the  data to be labeled(input_data_path)
input_data_path = "/Users/jisha/Desktop/Sarcasm_Final/Exploratory_Data_Analysis/final_bitcoin_data.csv"
# Path where the results to be saved
output_data_path = "/Users/jisha/Desktop/Sarcasm_Final/Prompt_Sensitivity/outputs.csv"

In [ ]:
# read the input data to a pandas dataframe and check the information about its contents 
input_data = pd.read_csv(input_data_path)
input_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 906 entries, 0 to 905
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   label           906 non-null    int64 
 1   comment         906 non-null    object
 2   author          906 non-null    object
 3   subreddit       906 non-null    object
 4   score           906 non-null    int64 
 5   ups             906 non-null    int64 
 6   downs           906 non-null    int64 
 7   date            906 non-null    object
 8   created_utc     906 non-null    object
 9   parent_comment  906 non-null    object
dtypes: int64(4), object(6)
memory usage: 70.9+ KB


In [ ]:
# Check if the path exists, if yes then read the data to new_labeled_df and check if the column llama_temp1 exists, 
# else create a new dataframe by copying the contents from the input data and create a new colukn named llama_temp1

if os.path.exists(output_data_path):
    print("Found an existing file with the same name, continuing labeling...")
    prompt_sensitivity_results = pd.read_csv(output_data_path)
    
    if 'ds_temp0' not in prompt_sensitivity_results.columns:
        prompt_sensitivity_results['ds_temp0'] = pd.NA
else:
    print("File not found with the given name, creating a new file and starting labeling from the beginning")
    prompt_sensitivity_results = input_data.copy()
    prompt_sensitivity_results['ds_temp0'] = pd.NA

# Find the index from where the labeling should start. 
def index_finder(df):
    for index, value in df['ds_temp0'].items():
        if pd.isna(value):
            return index
        if value not in [0, 1, 0.0, 1.0]:
            return index
    return len(df)

start_index = index_finder(prompt_sensitivity_results)

if start_index >= len(prompt_sensitivity_results):
    print("\n The dataset is already fully labeled. No further labeling is required.")
else:
    print(f"\n Continuing labelling from row index: {start_index} out of {len(prompt_sensitivity_results)}")

Found an existing file with the same name, continuing labeling...

 The dataset is already fully labeled. No further labeling is required.


In [ ]:
def labeller(parent_comment, comment):
    prompt = f"""
    
    Act as an expert in Financial sentiment analysis, you are specialised in analysing social media data, more specifically about bitcoin.

    now your task is to analyse the relationship between two comments and find the replay comment is a sarcastic replay to the parent comment.
       
    Financial sarcasm is often subtle, relying on hyperbole, irony, or mocking specific market trends (e.g., "To the moon!" when a stock crashes, or thanking a CEO for losing money).

    Analyze the comments provided and  Determine if the response comment is sarcastic based on the context of the parent comment.

    Output rules:
    - Output ONLY a single digit 1 if the comment is sarcastic if not 0.
    - Do not include any other text, explanation, punctuation, or spaces.
    - Remember you cannot give anything other than 0 or 1
    - The Parent comment: "{parent_comment}"
    - The Reply comment: "{comment}"

    """
    try:
        # Call the deepseek-r1:8b model with temprature 0.0
        result = ollama.chat(
            model='deepseek-r1:8b', 
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.0}  # Keeps outputs deterministic
        )
        original_response = result['message']['content'].strip()
        
        # Extract the first clean number from the model's output
        if "1" in original_response:
            return 1
        elif "0" in original_response:
            return 0
        else:
            # Fallback in case it outputs text like "Sarcastic" or "Not sarcastic"
            return 1 if "sarcastic" in original_response.lower() else 0
            
    except Exception as e:
        raise RuntimeError(f"Ollama Call Error: {e}")

In [ ]:
# Strat labeling the data from the starting index given by the function index_finder
# Iterate through every unlabeled rows

if start_index < len(prompt_sensitivity_results):
    print("\nIterating through remaining unlabelled rows via local Ollama instance...")
    start_time = time.time()

    non_labelled_rows = prompt_sensitivity_results.iloc[start_index:]

    # Create the progress bar instance
    progress_bar = tqdm(non_labelled_rows.iterrows(), total=len(non_labelled_rows), miniters=25)

    for index, row in progress_bar:
        try:
            prediction = labeller(row['parent_comment'], row['comment'])
            prompt_sensitivity_results.at[index, 'ds_temp0'] = prediction

            # Update the progress bar text dynamically every 25 rows (without printing new lines)
            if index % 25 == 0:
                progress_bar.set_description(f"Processing Row {index}")

        except Exception as e:
            # Print errors so you don't miss failure warnings
            tqdm.write(f"Row {index} skipped due to error: {e}")
            continue

        # Save progress every 10 rows
        if index % 10 == 0:
            prompt_sensitivity_results.to_csv(output_data_path, index=False)

    # Close progress bar and complete final save
    progress_bar.close()
    prompt_sensitivity_results.to_csv(output_data_path, index=False)

    # Calculate execution metrics
    end_time = time.time()
    total_seconds = end_time - start_time
    hours, minutes, seconds = int(total_seconds // 3600), int((total_seconds % 3600) // 60), int(total_seconds % 60)

    # Final summary display
    print(f"Dataset labeling complete!")
    print(f"Total rows processed: {len(non_labelled_rows)}")
    print(f"Total time taken to label data:   {hours}h {minutes}m {seconds}s")
